# LC 23 — Merge K Sorted Lists
**Day-63 | Hard | Heap / Priority Queue**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> To find the global minimum among
k sorted lists, maintain a min-heap of size k — one entry per
list's current front node. Pop the min, advance that list, push
its next node. Never compare ListNode objects directly; use
a list index as a tiebreaker.
</div>

## Official Problem Statement

You are given an array of `k` linked-lists `lists`, each linked-list
is sorted in ascending order.
Merge all the linked-lists into one sorted linked-list and return it.

**Example 1:**
```
Input:  lists = [[1,4,5],[1,3,4],[2,6]]
Output: [1,1,2,3,4,4,5,6]
```
**Example 2:**
```
Input:  lists = []
Output: []
```
**Example 3:**
```
Input:  lists = [[]]
Output: []
```

**Constraints:**
- `k == lists.length`
- `0 <= k <= 10^4`
- `0 <= lists[i].length <= 500`
- `-10^4 <= lists[i][j] <= 10^4`
- Each `lists[i]` is sorted in ascending order.
- Sum of all `lists[i].length` <= `10^4`

## What This Is Actually Asking

We have k already-sorted streams of numbers and need to merge
them into one sorted stream — efficiently.
The naive approach of collecting all values and sorting them
costs O(N log N) where N is total nodes, losing the sorted
structure we already have.
The key observation: at every step, the next output node must
be the smallest of the k current front nodes.
A min-heap gives us that minimum in O(log k) instead of O(k),
dropping the total cost to O(N log k).

## Walk Through an Example by Hand

`lists = [[1,4,5], [1,3,4], [2,6]]`

```
Initialize heap with front of each list:
  heap = [(1,0,node1->4->5), (1,1,node1->3->4), (2,2,node2->6)]

Step 1: Pop (1,0,node). Append 1. Push node.next=(4,0,node4->5).
  result: [1]
  heap:   [(1,1,node1->3->4), (2,2,node2->6), (4,0,node4->5)]

Step 2: Pop (1,1,node). Append 1. Push (3,1,node3->4).
  result: [1,1]
  heap:   [(2,2,node2->6), (3,1,node3->4), (4,0,node4->5)]

Step 3: Pop (2,2,node). Append 2. Push (6,2,node6).
  result: [1,1,2]
  heap:   [(3,1,node3->4), (4,0,node4->5), (6,2,node6)]

Step 4: Pop (3,1). Append 3. Push (4,1,node4).
  result: [1,1,2,3]

... continue until heap empty ...
  final: [1,1,2,3,4,4,5,6]
```

## The Picture

```
k sorted lists feeding into a min-heap:

List 0:  1 --> 4 --> 5
              ^
List 1:  1 --> 3 --> 4
              ^         Min-Heap (size <= k)
List 2:  2 --> 6        +------------------+
              ^         | (1, idx=0, node) |
                        | (1, idx=1, node) |  <-- pop min
                        | (2, idx=2, node) |
                        +------------------+
                                |
                                v
                   Output list: 1 -> 1 -> 2 -> ...

Heap entry = (val, list_index, node)
  val:        for heap ordering
  list_index: tiebreaker (avoids comparing ListNode)
  node:       to find node.next after popping

Loop:
  while heap not empty:
    val, i, node = heappop(heap)
    append node to result
    if node.next:
        heappush(heap, (node.next.val, i, node.next))
```

## When To Use This Pattern

- When merging **k sorted sequences** (lists, arrays, streams),
  think min-heap of size k.
- When you need to **repeatedly extract the global minimum**
  from multiple sorted sources, think heap.
- When comparing objects that are **not directly comparable**
  (like ListNode), think heap with a numeric tiebreaker index.
- When the total number of elements N is large but k is small,
  think O(N log k) vs O(N log N) — the heap pays off.
- When streaming data arrives from multiple ordered sources
  (Kafka topics, DB cursors), think this heap merge pattern.

## The Approach

Seed a min-heap with the first node of each non-empty list,
storing tuples of `(val, list_index, node)` — the list_index
is critical to break ties without comparing ListNode objects.
Repeatedly pop the smallest element, attach it to a dummy
result list, and push that node's `next` (if it exists) back
into the heap.
When the heap is empty, every node has been processed and the
result list is fully sorted.

In [ ]:
from typing import List, Optional
import heapq


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def make_list(vals):
    dummy = ListNode(0)
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next


def to_list(head):
    r = []
    while head:
        r.append(head.val)
        head = head.next
    return r

In [ ]:
def test_harness(func):
    cases = [
        # (input_lists, expected, label)
        (
            [[1,4,5],[1,3,4],[2,6]],
            [1,1,2,3,4,4,5,6],
            "classic 3 lists"
        ),
        ([], [], "empty input"),
        ([[]], [], "one empty list"),
        ([[1]], [1], "single node"),
        ([[1,2,3]], [1,2,3], "one list"),
        ([[1,1],[1,1]], [1,1,1,1], "all same"),
        ([[-1,0,1],[-2,0,2]], [-2,-1,0,0,1,2], "negatives"),
    ]
    passed = 0
    for raw_lists, expected, label in cases:
        linked = [make_list(lst) for lst in raw_lists]
        result_head = func(linked)
        result = to_list(result_head)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status} [{label}]: "
                f"got {result}, expected {expected}"
            )
    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")

In [ ]:
def mergeKLists(
    lists: List[Optional[ListNode]]
) -> Optional[ListNode]:
    """
    Merge k sorted linked lists using a min-heap.

    Strategy:
        - Create dummy head for result list.
        - Seed heap: for each non-None list head,
          push (node.val, list_idx, node).
        - While heap:
            val, i, node = heappop(heap)
            cur.next = node; cur = cur.next
            if node.next: heappush((node.next.val,i,node.next))
        - Return dummy.next

    Args:
        lists: List of heads of k sorted linked lists

    Returns:
        Head of merged sorted linked list

    Time:  O(N log k) — N total nodes, heap size k
    Space: O(k) — heap stores at most k entries
    """
    # Debug: print list values
    print(f"[debug] k={len(lists)} lists")
    for i, head in enumerate(lists):
        print(f"  list[{i}] = {to_list(head)}")

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(mergeKLists)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Collect + sort | O(N log N) | O(N) | Ignores existing order |
| Merge pairs (naive) | O(kN) | O(1) | Each merge touches all |
| Divide & conquer | O(N log k) | O(log k) | Recursive pairing |
| **Min-heap** | **O(N log k)** | **O(k)** | **Iterative, clear** |

Where N = total number of nodes across all lists.

## Real World Connection

At AWS, merging k sorted lists is exactly what happens when
combining sorted outputs from k parallel map tasks in a
MapReduce job — each reducer receives pre-sorted partitions
and merges them with this heap pattern.
Citi's trading systems merge order books from multiple
exchanges, each arriving as a sorted stream; the heap ensures
the consolidated book is always correct with minimal latency.
In data engineering, this pattern appears in external sort
algorithms where sorted runs from disk are merged in memory
one chunk at a time.
The O(N log k) vs O(N log N) difference matters at scale:
merging 1000 sorted streams of 10M records each is vastly
faster with k=1000 in the exponent versus N=10B.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra